In [1]:
!python --version

Python 3.13.9


In [2]:
import pandas as pd
import numpy as np

from scipy.sparse import coo_matrix
from lightfm import LightFM
from sklearn.metrics.pairwise import cosine_similarity

D:\Apps\anaconda\Data\envs\lightfm_env\lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [3]:
# Load processed datasets

interactions = pd.read_csv(r"C:\Users\sahil\AAdler\Revenue Growth & Personalization Analytics Package\Data\processed\interaction_matrix.csv")
products = pd.read_csv(r"C:\Users\sahil\AAdler\Revenue Growth & Personalization Analytics Package\Data\processed\product_features.csv")

print("Data loaded")

Data loaded


In [4]:
# Map IDs

user_ids = interactions["customer_id"].unique()
item_ids = interactions["product_id"].unique()

user_map = {u: i for i, u in enumerate(user_ids)}
item_map = {i: j for j, i in enumerate(item_ids)}

# Convert to indices

interactions["user_idx"] = interactions["customer_id"].map(user_map)
interactions["item_idx"] = interactions["product_id"].map(item_map)

# Create sparse matrix

interaction_matrix = coo_matrix(

    (
        interactions["purchase_count"],
        (interactions["user_idx"], interactions["item_idx"])
    )

)

In [ ]:
model = LightFM(

    no_components=30,
    learning_rate=0.05,
    loss="warp"

)

model.fit(

    interaction_matrix,
    epochs=20,
    num_threads=4

)

print("Collaborative filtering model trained")

In [ ]:
def recommend_products(user_id, top_n=5):

    if user_id not in user_map:
        return []

    user_index = user_map[user_id]

    scores = model.predict(

        user_index,
        np.arange(len(item_ids))

    )

    top_items = np.argsort(-scores)[:top_n]

    recommended_products = [

        item_ids[i] for i in top_items

    ]

    return recommended_products

In [ ]:
# Create product feature matrix

product_matrix = pd.get_dummies(

    products[["category", "price_band"]]

)

similarity_matrix = cosine_similarity(product_matrix)

In [ ]:
product_index = {

    pid: idx for idx, pid in enumerate(products["product_id"])

}

def similar_products(product_id, top_n=5):

    idx = product_index[product_id]

    scores = similarity_matrix[idx]

    similar_idx = np.argsort(-scores)[1:top_n+1]

    return products.iloc[similar_idx]["product_id"].tolist()

In [ ]:
popular_products = interactions.groupby(

    "product_id"

)["purchase_count"].sum().sort_values(

    ascending=False

).head(10).index.tolist()

In [ ]:
def hybrid_recommendation(user_id):

    if user_id in user_map:

        return recommend_products(user_id)

    else:

        return popular_products[:5]

In [ ]:
recommendations = []

for user in user_ids:

    recs = recommend_products(user, 5)

    for product in recs:

        recommendations.append({

            "customer_id": user,
            "recommended_product": product

        })

recommendation_df = pd.DataFrame(recommendations)

recommendation_df.to_csv(

    "../data/output/recommended_products.csv",
    index=False

)

print("Recommendations generated")